In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dask.distributed import Client, LocalCluster

client = Client() # Note that `memory_limit` is the limit **per worker**.
# n_workers=4,
#                 threads_per_worker=1,
#                 memory_limit='3GB'
client # If you click the dashboard link in the output, you can monitor real-time progress and get other cool visualizations.

In [ ]:
import copy
import sys
import xarray as xr
import numpy as np
import dask.array as da

import matplotlib.pyplot as plt
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
from holoviews import opts
import scipy.constants

sys.path.append("..")
import processing_dask as pr
import gps_processing as gps
import Geostacking as geostack
import plot_dask
import processing as old_processing

# for parsing GPS and stdout log files
import re
import pandas as pd

sys.path.append("../../preprocessing/")
from generate_chirp import generate_chirp

In [ ]:
plt.rcParams.update({'font.size': 16})

In [ ]:
def parse_stdout_times(line):
    # Extract the timestamp from lines that contain specific phrases
    if '[START] Beginning main loop' in line:
        pattern = r'(\d+\.\d+)'
        match = re.search(pattern, line)
        if match:
            return float(match.group(1))
            
    elif '[TX] Closing file' in line:
        pattern = r'(\d+\.\d+)'
        match = re.search(pattern, line)
        if match:
            return float(match.group(1))

    return None

In [ ]:
def start_and_stop_from_log(logstring):
    lines = logstring.split("\n")
    results = []
    for line in lines:
        parsed = parse_stdout_times(line.strip())
        if parsed:
            results.append(parsed)
    
    return {"start": results[0], "stop": results[1]}

In [ ]:
def get_distance_along_track(df):
    displacements = np.linalg.norm(df.iloc[1:,:].loc[:, ["ecefx", "ecefy", "ecefz"]].to_numpy() - \
    df.iloc[:-1,:].loc[:, ["ecefx", "ecefy", "ecefz"]].to_numpy(), axis=1)
    return np.cumsum(displacements)

### Open and resave file

In [ ]:
# file path to data and configs

prefix = "/Volumes/Extreme SSD/orca_paper/20240226_105437" # Anna loopback spectrogram, no phase dithering
prefix = "/home/thomas/Documents/StanfordGrad/RadioGlaciology/drone/radar_data/20230723-summit-day3-2start/20230723_104248"
prefix = "/Users/thatch/Projects/SORA/before-and-after-test/20251002_175426"
prefix = "/Volumes/sora-sshfs/20251127_142119" # first shot - only went to 1km
prefix = "/Volumes/sora-sshfs/20251127_143302" # second shot, longer rx window, 4km
prefix = "/Volumes/sora-sshfs/20251127_143818" # thrid shot - Mt Rossman
prefix = "/Volumes/sora-sshfs/20251127_144118" # fourth shot - straight down - 1 sec or so
prefix = "/Volumes/sora-sshfs/20251127_144528" # fifth shot - straight down - 10 sec or so
prefix = "/Users/thatch/Projects/SORA/data/20251127_144528"

prefix = "/Volumes/sora-sshfs/20251130_124406" # Nov 30 - first LO - OOPS! just attenuators! no cabl

#path = "/Volumes/sora-sshfs/"
#path = "/tmp/SORA-1130/"
test = "20251130_125618"  # first real LO at CI with attns and cable!! looks pretty good!
test = "20251130_155708" # lpda shot outside UG after SSD issue
#test = "20251130_160010" # lpda dragged by ~10 meters
#test = "20251130_175353" # |  | pol 10 s
#test = "20251130_175636" # x pol 10 s
#test = "20251130_175854" # --  -- pol 10 s
#test  = "20251130_181559" #10 sec loop;back
#test = "20251130_183246" # 10 sec -- -- pol
#test = "20251130_183459" # 10 sec |  | pol
#test = "20251130_234658" # ref cable lo, 20db attn, rx only
#test = "20251201_000404" # suspected bad cable, white tape
#test = "20251201_000952" # suspected good cable, no tape

#test = "20251201_004608"   # suspected good
#test = "20251201_005419"   # suspected bad
#test = "20251201_005945"   # reference cable

#test = "20251201_011014"   # all up test in camp
#test = "20251201_011420"    # and again
#test = "20251201_011703"    # and agaibn
#test = "20251201_011829" # lower tx and rx gain
#test = "20251201_012005" # even less tx
#test = "20251201_012130" # even less tx gain

#test = "20251201_125531" # low gain xpol
#test = "20251201_125646" # reg gain xpol
#test = "20251201_125729" # reg gain same pol

#test = "20251201_144352" # rfspace antennas

# 4 rfspace antenna tests
#test = "20251201_155939" # 450, rtk on
#test = "20251201_160032" # 390, rtk on
#test = "20251201_160451" # 390 rtk off
#test = "20251201_160533" # 450, rtk off

#test = "20251201_172432" # 450, rtk off, lpdas
#test = "20251201_172921" # "" +5 tx gain

path = "/Volumes/CI1/"
test = "20251201_190207" # static shot at 1km upstream
#test  = "20251201_191042" # first traveerse, 200m
#test = "20251201_192052" # second traverse, 200m 
#test = "20251201_202215" # 1km traverse upstream, ending at camp

#dec 2
# parallel plane pol, tx gain50
# inplane pol, tx gain59
test = "20251202_124842" # inplane pol, tx gain50  - start of test
#test = "20251202_125159" # inplane, txgain 45
#test=  "20251202_125429" # inplane, txgain 40
#test = "20251202_125613" # ", txgain 35
#test = "20251202_125843" # ", txgain 32
#test = "20251202_130023" # ", 27 db
#test = "20251202_131542"  # same, farther from lehman
#test = "20251202_131734" # 22 db tx
test = "20251202_131932" # "", lower rx gain
test = "20251202_132228" # lower rx gain       - still clipping
test = "20251202_132409" # 10 db rx gain
#test = "20251202_132542" # same ^
#test = "20251202_134647" # same but with rfspace antennas
#test = "20251202_134809" # higher tx rx gain

test = "20251202_180847" # shot 3k afternoon dec 2
#test = "20251202_181055" # same +3db tx
#path = "/tmp/SORA-1202/"
#test = "20251202_183128" # ~1km traverse from shot 3k
#test = "20251202_183653" # station at line upstream of camp
#test = "20251202_183833" # same but +3db 
test = "20251202_190714" # long transect -- sea water reflection!!!!
#test = "20251202_190820" # statijnary at end of tsect sea water!!!!!
#test = "20251202_999999"

#path = "/tmp/SORA-1203/"
#test = "20251203_182424" # good conf from yesterday
#test  ="20251203_182518" # same but 5us chrip

#test = "20251203_190524"
#test = "20251203_194038" # long downstream transect dec 3, 7 kph
#test = "20251203_195317" # upstream transect to camp - 12 kph

# rf space antenna test - hopefully less clipping
#test = "20251203_210905" # reference with lpdas
#test = "20251203_211322" # same conf with rf spaces tx gain 56
#test = "20251203_211458" # rf space, txgain 46
#test = "20251203_211645" # ", txgan 36
#test =  "20251203_211834" # "" rxgain 26
#test = "20251203_212053" # rxgain 16
#test = "20251203_212350" # "", parallel pol
#test= "20251203_212514" # "", xpol
#test = "20251203_212658" # still xpol, big gains
#test = "20251203_212821" # parallel plane pol, big gains

# dec 4
#test = "20251204_212545"
test = "20251204_215913" # 5km down das line
#test = "20251204_221135" # 2km side leg
#test = "20251204_224207" # 4 km upstream traverse on smart solo like (near smart solo 4016)


# Dec 5
#test = "20251205_154838" # 4 km down stream, 12 kph
#test = "20251205_161625" # 2km up starboard wing, 12 kph then 7 kph
#test = "20251205_162708" # diagonal down across GL, 12? kph
#test = "20251205_164609" # 3 km up stryde line

# rx gain tests at camp
#test  ="20251205_170039" # reference = tx gain 56 db, rx gain 36 db
#test = "20251205_170508" # 26 db
#test = "20251205_170900" # rx gain 166 !!!!!
#test = "20251205_171306" # rx gain 16 # 10 db SNR
#test = "20251205_171525" # rx gain 6
#test = "20251205_171810" # rx gain 8

# Dec 6 - low rx gain!
# back 9
#test = "20251206_134321"
#test = "20251206_140706" # back 4
#test = "20251206_142331"

# left wing
#test = "20251206_161447" # western boundary
#test = "20251206_162356" # ocean segment
#test = "20251206_163720" # diagonal upstream western GL

# dec 7 left wing
#test = "20251207_153056" # down das line
#test = "20251207_160617" # up western boundary
#test = "20251207_162558" # down the dogleg
#test = "20251207_164411" # up the stride line

# dec 7 opres test - no rfpa., 350 mhz
#test = "20251207_170138"
#test = "20251207_170444" # 70 txgain
#test = "20251207_170748" #  + 4db rx gain
#test = "20251207_171023" # + 3 db rx

#test = "20251207_214057" # test at start of run, rx gain36
#test = "20251207_214310" # rx gain26

# Dec 7-8, opres overnight
#test = "20251208_133603" # great bed return! no real brightness variation... phase could be interesting!

# dec 8
#test = "20251208_155425" # down das line, clipped tucker
#test = "20251208_162754" # starboard star offline segment

# afternoon figure 8 track
#test = "20251208_214018" # igloo to ug04 (?)
#test = "20251208_215131" # gnss to right angle upstream
#test = "20251208_220414" # crossing line
#test = "20251208_221734" # upstream on west side
#test = "20251208_222354" #igloo to ug03

# dec 9 !
#test = "20251209_160157" # ug03 to ug01, 350
#test = "20251209_162232" # ug01 to ug08 (out of box), 350
#test = "20251209_162544" # check at 450 at ug08, txgain 57

# gl laps


#test = "20251209_200750" # leg 3 - down at 350, 58 db

prefix = path + test

# resave data as zarr for dask processing
zarr_base_location="/Users/thatch/Projects/SORA/before-and-after-test/test_tmp_zarr_cache/"
zarr_path = pr.save_radar_data_to_zarr(prefix, zarr_base_location=zarr_base_location)

# open zarr file, adjust chunk size to be 10 MB - 1 GB based on sample rate/bit depth
raw = xr.open_zarr(zarr_path, chunks={"pulse_idx": 1000})

In [ ]:
gpsdf = gps.extract_locations_from_gpspipe( raw.gpspipe_log )

In [ ]:
res = start_and_stop_from_log(raw.stdout_log)
res["stop"] - res["start"]

In [ ]:
(raw.slow_time[-1] - raw.slow_time[1]).compute().values

In [ ]:
gpsdf["slow_time"] = gpsdf["unix_time"] - res["start"]

In [ ]:
georaw = geostack.add_distance_to_xarray(raw, 
                                         raw.slow_time.compute(), 
                                         gpsdf["slow_time"].iloc[1:],
                                         get_distance_along_track(gpsdf))

### Enter processing parameters

In [ ]:
#zero_sample_idx = 36 # X310, fs = 20 MHz
#zero_sample_idx = 63 # X310, fs = 50 MHz
zero_sample_idx = 159 # B205mini, fs = 56 MHz
#zero_sample_idx = 166 # B205mini, fs = 20 MHz

nstack = 1000 # number of pulses to stack
lenstack = 5 # number of meters to stack over

modify_rx_window = False # set to true if you want to window the reference chirp only on receive, false uses ref chirp as transmitted in config file
rx_window = "blackman" # what you want to change the rx window to if modify_rx_window is true

dielectric_constant = 3.17# ice (air = 1, 66% velocity coax = 2.2957)
#dielectric_constant = 2.2957 # COAX (air = 1, 66% velocity coax = 2.2957)
#dielectric_constant = 1.0
sig_speed = scipy.constants.c / np.sqrt(dielectric_constant)

### Generate reference chirp

In [ ]:
if modify_rx_window:
    config = copy.deepcopy(raw.config)
    config['GENERATE']['window'] = rx_window
else:
    config = raw.config

chirp_ts, ref_chirp = generate_chirp(config)

### View raw pulse in time domain to check for clipping

In [ ]:
single_pulse_raw = raw.radar_data[{'pulse_idx': 100}].compute()
plot1 = np.real(single_pulse_raw).hvplot.line(x='fast_time', color='red') * np.imag(single_pulse_raw).hvplot.line(x='fast_time')

plot1 = plot1.opts(xlabel='Fast Time (s)', ylabel='Raw Amplitude', title=f"Pulse 100 from {test}")
plot1

### Clean and stack data

In [ ]:
geostacked = pr.fill_errors(georaw, error_fill_value=0.0) # fill receiver errors with 0s
geostacked = geostack.stack(geostacked, lenstack)

In [ ]:
stacked = pr.fill_errors(raw, error_fill_value=0.0) # fill receiver errors with 0s
stacked = pr.stack(stacked, nstack) # stack 

### Pulse compress data

In [ ]:
compressed = pr.pulse_compress(stacked, ref_chirp,
                               fs=stacked.config['GENERATE']['sample_rate'],
                               zero_sample_idx=zero_sample_idx,
                               signal_speed=sig_speed)

geocompressed = pr.pulse_compress(geostacked, ref_chirp,
                               fs=stacked.config['GENERATE']['sample_rate'],
                               zero_sample_idx=zero_sample_idx,
                               signal_speed=sig_speed)

compressed_power = xr.apply_ufunc(
    lambda x: 20*np.log10(np.abs(x)),
    compressed,
    dask="parallelized"
)

geocompressed_power = xr.apply_ufunc(
    lambda x: 20*np.log10(np.abs(x)),
    geocompressed,
    dask="parallelized"
)

### View 1D pulse compressed data

In [ ]:

plot1D = np.mean(compressed_power.radar_data, axis=0).hvplot.line(label="Mean of All Pulses")
plot1D = plot1D * compressed_power.radar_data[0,:].hvplot.line(label="First Pulse")
plot1D = plot1D * compressed_power.radar_data[-1,:].hvplot.line(label="Last Pulse")

# relevant options: xlim(-80,1000)

plot1D = plot1D.opts(xlabel='Reflection Distance (m)', ylabel='Return Power (dB)')
plot1D = plot1D.opts(title=test+" (ϵ="+str(dielectric_constant)+")")
plot1D.opts(xlim=(-50,2300), ylim=(-120, -25), show_grid=True)

### View 2D pulse compressed data (radargram)

In [ ]:
# USING HOLOVIEWS (sometimes breaks)
plot2D = compressed_power.swap_dims({'pulse_idx': 'slow_time', 'travel_time': 'reflection_distance'}).hvplot.quadmesh(x='slow_time', cmap='inferno', flip_yaxis=True)
# relevant options: ylim=(100,-50), clim=(-90,-40)

plot2D.opts(xlabel='Slow Time (s)', ylabel='Depth (m)', clabel='Return Power (dB)')
plot2D.opts(ylim=(-10, 2000), clim=(-115, -70))

In [ ]:
# Select data at reflection_distance ≈ 1616
depth_m = 1590

line_data = compressed_power.swap_dims({
    'pulse_idx': 'slow_time', 
    'travel_time': 'reflection_distance'
}).sel(reflection_distance=depth_m, method='nearest')

# Plot as a line
plot1D = line_data.hvplot.line(x='slow_time')
plot1D.opts(xlabel='Slow Time (s)', ylabel='Return Power (dB)')


In [ ]:
# USING HOLOVIEWS (sometimes breaks)
geoplot2D = geocompressed_power.swap_dims({'travel_time': 'reflection_distance'}).hvplot\
    .quadmesh(x='distance_bin', cmap='inferno', flip_yaxis=True, width=1000)
# relevant options: ylim=(100,-50), clim=(-90,-40)

geoplot2D.opts(xlabel='Distance along track (m)', ylabel='Depth (m)', clabel='Return Power (dB)')
geoplot2D.opts(ylim=(-10, 2000), clim=(-100, -65))

In [ ]:
camp_latitude = -79.58937
(gpsdf.loc[:, ["slow_time", "lat"]].hvplot.line(x='slow_time') * hv.HLine(camp_latitude) + \
    gpsdf.loc[:, ["slow_time", "lon"]].hvplot.line(x='slow_time', color='red')).cols(1)

In [ ]:
# USING MATPLOTLIB (sometimes takes a while)
fig, ax = plt.subplots(1,1, figsize=(10,6), facecolor='white')

p = ax.pcolormesh(compressed_power.slow_time, \
                  compressed_power.reflection_distance, \
                  compressed_power.radar_data.transpose(), \
                  shading='auto', cmap='inferno',\
                 # vmin = -140, vmax=-60
                 )
ax.invert_yaxis()
clb = fig.colorbar(p, ax=ax)
clb.set_label('Return Power (dB)')
ax.set_xlabel('Slow Time (s)')
ax.set_ylabel('Distance to Reflector (m)')
# relevant options: ax.set_ylim=(100,-50), ax.set_xlim=(0, 1), vmin=-90, vmax=40
ax.set_ylim(2000, -50)

### View spectrogram of stacked data

In [ ]:
inpt = raw
inpt["radar_data"].shape

In [ ]:
num_presums = raw.attrs["config"]["CHIRP"]["num_presums"]

In [ ]:
# data = stacked["radar_data"].to_numpy()
n = 10
normalize = True

pulse = pr.stack(inpt, n)[{'pulse_idx':0}]["radar_data"].to_numpy()

f, t, S = scipy.signal.spectrogram(
    pulse,
    fs=raw.attrs["config"]["GENERATE"]["sample_rate"],
    window='flattop',
    nperseg=128,
    noverlap=64,
    scaling='density', mode='psd',
    return_onesided=False
)

if normalize:
    S /= np.max(S)

In [ ]:
fig, ax = plt.subplots(facecolor='white', figsize=(10,6))
freq_mhz = (np.fft.fftshift(f) + raw.attrs['config']['RF0']['freq']) / 1e6
pcm = ax.pcolormesh(t, freq_mhz, 10*np.log10(np.abs(np.fft.fftshift(S, axes=0))), shading='nearest') #  vmin=-420, vmax=-200
clb = fig.colorbar(pcm, ax=ax)
clb.set_label('Power [dB]')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Frequency [MHz]')
#ax.set_title(f"Spectrogram of received data with n_stack={n}")
ax.text(0, 1.05, prefix.split("/")[-1] + "\n" + f"n_stack * num_presums = {n * num_presums}", horizontalalignment='left', verticalalignment='center', transform=ax.transAxes, fontdict={'size': 12})
fig.tight_layout()
plt.show()

### View Power Spectrum of All Received Data

In [ ]:
single_stack = pr.stack(raw, raw.radar_data.shape[1])

data_rx_fft = da.fft.fft(raw.radar_data, axis=0) / raw.radar_data.shape[0]
stacked_fft = da.fft.fft(stacked.radar_data, axis=0) / stacked.radar_data.shape[0]
full_fft = da.fft.fft(single_stack.radar_data, axis=0) / single_stack.radar_data.shape[0]

data_rx_fft_pwr = 20*da.log10(da.abs(data_rx_fft))
stacked_fft_pwr = 20*da.log10(da.abs(stacked_fft))
full_fft_pwr = 20*da.log10(da.abs(full_fft))

#data_rx_fft_pwr.shape

In [ ]:
# fig, axs = plt.subplots(2,1)
fig, axs = plt.subplots(facecolor='white', figsize=(10,6))
freqs = np.fft.fftshift(np.fft.fftfreq(data_rx_fft_pwr.shape[0], d=1/raw.config['GENERATE']['sample_rate']))
axs.plot(freqs/1e6, np.fft.fftshift(data_rx_fft_pwr[:,0]), label='Single Pulse')
axs.plot(freqs/1e6, np.fft.fftshift(stacked_fft_pwr[:,0]), label='Single Stack')
axs.plot(freqs/1e6, np.fft.fftshift(full_fft_pwr[:,0]), label='Full File')
axs.set_xlabel('Frequency [MHz]')
axs.set_ylabel('Power [dB]')
axs.set_title('Spectrum -- Power')
axs.grid()
axs.legend()

# axs[1].plot(freqs/1e6, np.fft.fftshift(np.angle(data_rx_fft[:,0])))
# axs[1].plot(freqs/1e6, np.fft.fftshift(np.angle(stacked_fft[:,0])))
# axs[1].plot(freqs/1e6, np.fft.fftshift(np.angle(full_fft[:,0])))
# axs[1].set_xlabel('Frequency [MHz]')
# axs[1].set_ylabel('Phase [rad]')
# axs[1].set_title('Spectrum -- Phase')
# axs[1].grid()
# fig.tight_layout()